# Лекция 1. Основы глубокого обучения — код

Код из главы 4 лекции: нейрон, полносвязный слой, обучение сети 2–2–1 на XOR и численная проверка градиентов. Запускайте ячейки по порядку сверху вниз.

## 4.2. Нейрон на NumPy — «в лоб»

$$z = \mathbf{w}^{\top}\mathbf{x} + b, \qquad y = f(z)$$

In [ ]:
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

class Neuron:
    """Один нейрон: n_inputs входов, сигмоидная активация."""
    def __init__(self, n_inputs):
        # малые случайные веса — ломаем симметрию (см. главу 3.8)
        self.w = np.random.randn(n_inputs) * 0.1
        self.b = 0.0

    def forward(self, x):
        z = np.dot(self.w, x) + self.b   # z = w·x + b
        return sigmoid(z)                # y = f(z)

**Что произошло.** Ячейка ничего не печатает: она только определяет функцию `sigmoid` и класс `Neuron`. Веса создаются случайными (`randn * 0.1`) — маленькими, чтобы нейроны не были одинаковыми.

Проверим на числах из главы 3: $\mathbf{x} = (1, 0)$, $w = (0.5, -0.3)$, $b = 0.1$ → $\sigma(0.6) \approx 0.646$.

In [ ]:
n = Neuron(2)
n.w = np.array([0.5, -0.3]); n.b = 0.1
print(n.forward(np.array([1.0, 0.0])))   # 0.6457 — совпало с ручным расчётом

**Результат: `0.6457`.** Мы вручную подставили веса из главы 3.5, поэтому случайная инициализация здесь не важна:

$$z = 0.5\cdot 1 + (-0.3)\cdot 0 + 0.1 = 0.6, \qquad \sigma(0.6) \approx 0.6457$$

Это ровно то число, которое мы считали на доске. Вход $x_2 = 0$, поэтому вес $w_2 = -0.3$ на ответ не повлиял.

## 4.3. Полноценный слой с прямым и обратным проходом

$$\frac{\partial L}{\partial \mathbf{Z}} = \frac{\partial L}{\partial \mathbf{A}} \odot f'(\mathbf{Z}), \qquad \frac{\partial L}{\partial \mathbf{W}} = \mathbf{X}^{\top}\frac{\partial L}{\partial \mathbf{Z}}, \qquad \frac{\partial L}{\partial \mathbf{X}} = \frac{\partial L}{\partial \mathbf{Z}}\,\mathbf{W}^{\top}$$

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_deriv(a):
    """Производная сигмоиды, выраженная через её же значение: a*(1-a)."""
    return a * (1.0 - a)

class DenseLayer:
    """Полносвязный слой: n_inputs -> n_neurons, активация sigmoid."""
    def __init__(self, n_inputs, n_neurons):
        # инициализация Ксавье/Глоро (Xavier/Glorot initialization):
        # масштаб случайных весов зависит от размера ОБОИХ измерений слоя
        # (n_inputs и n_neurons), чтобы дисперсия активаций не менялась
        # ни при прямом, ни при обратном проходе через слой
        self.W = np.random.randn(n_inputs, n_neurons) * np.sqrt(2.0 / (n_inputs + n_neurons))
        self.b = np.zeros(n_neurons)

    def forward(self, X):
        # X: (batch, n_inputs); A: (batch, n_neurons)
        self.X = X                      # запоминаем вход — нужен для backward
        self.Z = X @ self.W + self.b    # Z = XW + b, как в главе 2
        self.A = sigmoid(self.Z)        # A = f(Z)
        return self.A

    def backward(self, dA):
        # dA: градиент потери по ВЫХОДУ слоя, (batch, n_neurons)
        dZ = dA * sigmoid_deriv(self.A)            # dL/dZ = dA ⊙ f'(Z)
        self.dW = self.X.T @ dZ                    # dL/dW = Xᵀ dL/dZ
        self.db = dZ.sum(axis=0)                   # dL/db
        return dZ @ self.W.T                       # dL/dX -> дальше, к предыдущему слою

    def update(self, lr):
        self.W -= lr * self.dW
        self.b -= lr * self.db

**Что произошло.** Снова только определения, вывода нет. Обратите внимание: `forward` **запоминает** `X` и `A` в атрибутах слоя — они понадобятся в `backward`. Поэтому `backward` всегда вызывается после `forward` на тех же данных.

## 4.4. Обучаем сеть 2–2–1 на XOR

Один нейрон XOR не решает (глава 1.3). Проверим, справится ли сеть 2–2–1.

In [ ]:
# --- Данные: все четыре комбинации XOR ---
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]], dtype=float)
y = np.array([[0],
              [1],
              [1],
              [0]], dtype=float)

# --- Сеть 2-2-1 ---
np.random.seed(3)   # фиксируем зерно (seed) генератора псевдослучайных
                    # чисел для воспроизводимости результата
layer1 = DenseLayer(2, 2)   # скрытый слой
layer2 = DenseLayer(2, 1)   # выходной слой

lr = 0.5  # learning rate — скорость обучения η
loss_history = []
for epoch in range(20000):
    # 1. Прямой проход (глава 2)
    A1 = layer1.forward(X)
    A2 = layer2.forward(A1)

    # 2. Потери: MSE (глава 3.2)
    loss = np.mean((A2 - y) ** 2)
    loss_history.append(loss)

    # 3. Обратный проход (глава 3.5–3.6)
    dA2 = 2 * (A2 - y) / len(X)   # производная MSE по выходу сети
    dA1 = layer2.backward(dA2)    # градиент проходит выходной слой
    layer1.backward(dA1)          # ...и скрытый слой

    # 4. Шаг градиентного спуска (глава 3.3)
    layer1.update(lr)
    layer2.update(lr)

    if epoch % 2000 == 0:
        print(f"эпоха {epoch:6d} | loss = {loss:.4f}")

print("Предсказания:")
print(np.round(layer2.forward(layer1.forward(X)), 3))

**Как читать вывод.**

- **Эпоха 0, loss ≈ 0.25.** Необученная сеть выдаёт на всех входах примерно 0.5, а правильные ответы — 0 или 1, поэтому ошибка на каждом примере ≈ $(0.5)^2 = 0.25$. Это уровень «случайного угадывания».
- **Эпохи 0–1750: плато.** Потери почти не меняются — скрытые нейроны ещё не нашли полезного представления, градиенты маленькие.
- **Эпохи ~1750–2800: резкий спад** с 0.2 до 0.01. Скрытый слой «перерисовал» пространство так, что классы стали линейно разделимы, и выходной нейрон быстро подстроился.
- **Дальше — медленный хвост** до 0.0003: сеть уже отвечает правильно и лишь «уточняет» уверенность.

**Предсказания** `0.017, 0.98, 0.984, 0.015` — округляем по порогу 0.5 и получаем `0, 1, 1, 0`, то есть ровно XOR. Точных 0 и 1 не будет никогда: сигмоида их не достигает, только приближается.

**Что выучил скрытый слой** (при `seed=3`): первый нейрон срабатывает почти везде, кроме входа (0, 1), второй — только на (1, 0). Выходной нейрон комбинирует их с весами разного знака — так из двух линейных границ получается нелинейная. Посмотреть веса можно, напечатав `layer1.W`, `layer1.b`, `layer2.W`.

Кривая потерь по эпохам:

In [ ]:
import matplotlib.pyplot as plt
plt.plot(loss_history)
plt.xlabel('эпоха'); plt.ylabel(r'$\mathcal{L}$'); plt.yscale('log')
plt.title('Кривая обучения'); plt.show()

**Как читать график.** Ось $y$ — в логарифмическом масштабе, иначе всё после 3000-й эпохи слилось бы с нулём. Видны те же три фазы: плато около 0.25, обрыв вниз, длинный пологий хвост. Если бы кривая росла или скакала — первым делом нужно уменьшать `lr`.

## 4.5. Проверка градиентов численно

$$f'(w) \approx \frac{f(w + \varepsilon) - f(w - \varepsilon)}{2\varepsilon}, \qquad \varepsilon \sim 10^{-7}$$

Если числа совпадают с точностью до ~1e-6 (в относительной мере) — backprop корректен.

In [ ]:
def loss_fn():
    out = layer2.forward(layer1.forward(X))
    return np.mean((out - y) ** 2)

# 1. аналитический градиент — пересчитываем в ТЕКУЩЕЙ точке:
#    layer1.dW из цикла обучения посчитан ДО последнего update
out = layer2.forward(layer1.forward(X))
layer1.backward(layer2.backward(2 * (out - y) / len(X)))

# 2. численный градиент для веса W1[0, 0]
eps = 1e-7
w_old = layer1.W[0, 0]
layer1.W[0, 0] = w_old + eps; L_plus  = loss_fn()
layer1.W[0, 0] = w_old - eps; L_minus = loss_fn()
layer1.W[0, 0] = w_old                     # возвращаем вес на место
num_grad = (L_plus - L_minus) / (2 * eps)

print(f"численный:     {num_grad:.10f}")
print(f"аналитический: {layer1.dW[0, 0]:.10f}")

**Результат.** Численный и аналитический градиенты совпали до 10 знаков после запятой — значит `backward` в `DenseLayer` написан правильно.

- **Почему градиент такой маленький** (≈ $-4\cdot10^{-5}$)? Сеть уже обучена и стоит почти на дне «долины», где поверхность потерь пологая. На необученной сети градиенты были бы на порядки больше.
- **Знак минус** означает: если чуть увеличить `W1[0, 0]`, потеря ещё немного уменьшится — сеть могла бы учиться дальше.
- Если бы в `backward` была ошибка (например, забыли умножить на `sigmoid_deriv`), численный и аналитический градиенты различались бы в разы, а не в десятом знаке.

## 4.6. Попробуйте сами

- Поменяйте `np.random.seed(3)` на `np.random.seed(42)` в ячейке 4.4 — сеть «застрянет» на loss ≈ 0.126.
- Поменяйте `lr` и посмотрите, как меняется кривая потерь.